# Speech Command Classifier: SraVaani ASR → Laya Decision Model

This notebook builds a pipeline that:

1. Takes an audio clip (Hindi, English, or mixed) as input.
2. Transcribes it with **SraVaani-1.0** (ARTPARK-IISc), a 65-language Indic ASR model.
3. Feeds the transcript to **Laya** (`convaiinnovations/laya`, multilingual checkpoint), a typed-decision model, with a `choice` question over six labels: `left`, `right`, `forward`, `backward`, `stop`, `other`.
4. Applies a confidence threshold and a keyword safety net for the `stop` command.
5. Includes a small evaluation harness so you can measure accuracy on your own labeled clips before trusting this in production.

**Before you run this:**
- Visit https://huggingface.co/ARTPARK-IISc/SraVaani-1.0 and accept the gated-access conditions with your Hugging Face account.
- Generate a Hugging Face access token (read scope is enough): https://huggingface.co/settings/tokens
- Have a GPU available if you want reasonable latency; both models together are large (~1.6 GB) and will run on CPU, just slower.

**Memory note:** SraVaani (~900 MB, FP16) and Laya-multilingual (~322M params, mmBERT-base) are both loaded into memory at once and kept resident for the life of the process. Budget accordingly on constrained devices.


## 1. Environment setup

Install the packages this notebook needs. Restart the kernel afterward if `transformers` or `torch` were freshly installed.

In [1]:
%pip uninstall -y torch torchvision
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cpu
# Core ML stack
%pip install -q --upgrade transformers accelerate soundfile librosa

# Laya's local runtime (typed decision model)

%pip install -q --upgrade laya

# Hugging Face Hub client, for the gated SraVaani model
%pip install -q "huggingface_hub<2.0,>=1.23.0"


Found existing installation: torch 2.11.0+cpu
Uninstalling torch-2.11.0+cpu:
  Successfully uninstalled torch-2.11.0+cpu
Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 MB 2.2 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 89.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.1/114.1 kB 3.6

### 1.1 Confirm versions

Sanity-check that everything imported and print versions, so a stale cached package doesn't silently break the pipeline later.

In [2]:
import torch, transformers, sys

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA avail. :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))

try:
    import laya
    print("laya module loaded OK")
except ImportError as e:
    print("laya import failed:", e)


Python      : 3.13.15
PyTorch     : 2.14.1+cpu
Transformers: 5.18.0
CUDA avail. : False
laya module loaded OK


## 2. Authenticate with Hugging Face

SraVaani-1.0 is a **gated** model. You must have accepted its access conditions on the model page before this will work. Paste your token below, or set the `HF_TOKEN` environment variable beforehand and skip the prompt.


In [3]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN") or input("Paste your Hugging Face access token: ").strip()
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")


Logged in to Hugging Face Hub.


## 3. Load the ASR model (SraVaani-1.0)

~430M parameters, FP16, ~900 MB on disk. First run downloads the weights, which can take a while depending on your connection.


In [4]:
from transformers import AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"

print(f"Loading {ASR_REPO} on {DEVICE} ...")
asr_model = AutoModel.from_pretrained(
    ASR_REPO,
    trust_remote_code=True,
    token=HF_TOKEN,
).to(DEVICE).eval()

print("ASR model loaded.")


Loading ARTPARK-IISc/SraVaani-1.0 on cpu ...


config.json:   0%|          | 0.00/561 [00:00<?, ?B/s]

configuration_sravaani.py:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ARTPARK-IISc/SraVaani-1.0:
- configuration_sravaani.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_sravaani.py:   0%|          | 0.00/16.1k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ARTPARK-IISc/SraVaani-1.0:
- modeling_sravaani.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

ASR model loaded.


### 3.1 Quick ASR smoke test

Point this at any short local WAV/FLAC/MP3 file (16 kHz mono is safest) to confirm transcription works before wiring up the full pipeline.

In [5]:
!wget -O /content/test.wav "https://github.com/Kishan-Agarwal-28/automatic-wheelchair/raw/refs/heads/main/test.wav"

--2026-10-05 05:35:32--  https://github.com/Kishan-Agarwal-28/automatic-wheelchair/raw/refs/heads/main/test.wav
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Kishan-Agarwal-28/automatic-wheelchair/refs/heads/main/test.wav [following]
--2026-10-05 05:35:32--  https://raw.githubusercontent.com/Kishan-Agarwal-28/automatic-wheelchair/refs/heads/main/test.wav
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4503192 (4.3M) [audio/wav]
Saving to: ‘/content/test.wav’

/content/test.wav   100%[===================>]   4.29M  --.-KB/s    in 0.08s   

2026-10-05 05:35:33 (55.8 MB/s) - ‘/content/test.wav’ s

In [6]:
TEST_AUDIO_PATH = "/content/test.wav"  # <-- change this

def transcribe(path: str) -> str:
    """Run SraVaani ASR on a single audio file and return the transcript text."""
    hyps = asr_model.transcribe(path, return_hypotheses=True)
    return hyps[0].text

if os.path.exists(TEST_AUDIO_PATH):
    text = transcribe(TEST_AUDIO_PATH)
    print("Transcript:", text)
else:
    print(f"No file found at {TEST_AUDIO_PATH} -- update the path above and re-run this cell.")


/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


Transcript: आगे चलो पीछे आओ दाएं मुड़ो बाएं मुड़ो मूव फॉरवर्ड मूव राइट मूव लेफ्ट मूव बैग स्टॉप रुक जाओ


## 4. Load the decision model (Laya)

We use the **multilingual** checkpoint (mmBERT-base) since traffic here is Hindi, English, or code-mixed. Laya's `Router` can auto-select a checkpoint, but for short Hinglish commands the automatic language/script routing can be ambiguous, so we pin the multilingual model explicitly.


In [7]:
from laya import Router

router = Router(default="multilingual")
print("Laya router ready (default checkpoint: multilingual).")


Laya router ready (default checkpoint: multilingual).


### 4.1 Define the command schema

One `choice` question with six labels. The `criteria` strings give the model concrete cues in both languages — extend these with more transliterations/synonyms you expect from real users.


In [12]:
COMMAND_QUESTIONS = {
    "command": {
        "type": "choice",
        "instructions": (
            "The text below is a speech-to-text transcript of a short spoken movement "
            "command, in Hindi, English, or a mix of both. The transcript may contain "
            "transcription errors -- a word can be misheard as a phonetically or "
            "visually similar word (for example 'back' misheard as 'bag', or 'right' "
            "misheard as 'write'). Interpret the transcript charitably: pick the command "
            "whose intended meaning the speaker most likely meant, even if the exact "
            "transcribed word does not literally match. Only choose 'other' if no "
            "command is plausibly intended."
        ),
        "criteria": {
            "left":     "turn or move left. Expected/likely-misheard forms: बाएं, बायें, baayen, left, lift, laft",
            "right":    "turn or move right. Expected/likely-misheard forms: दाएं, दायें, daayen, right, write, rite",
            "forward":  "go ahead or straight. Expected/likely-misheard forms: आगे, aage, forward, foward, seedha, straight, straight",
            "backward": "go back or reverse. Expected/likely-misheard forms: पीछे, peeche, back, bag, backward, buckward, reverse",
            "stop":     "halt or stop moving. Expected/likely-misheard forms: रुको, रुक जाओ, ruko, ruk jao, stop, stopp, top",
            "other":    "the transcript does not plausibly correspond to any of the above commands",
        },
    }
}

print("Command schema defined:", list(COMMAND_QUESTIONS["command"]["criteria"].keys()))

Command schema defined: ['left', 'right', 'forward', 'backward', 'stop', 'other']


## 5. Full pipeline: audio → text → label

Combines ASR and Laya into one function, with:
- a **confidence threshold** (`CONF_MIN`) that forces low-confidence predictions to `other`,
- a **keyword safety net** for `stop`, since a missed stop command is the costliest failure mode in most command-driven systems.

Tune `CONF_MIN` and the keyword lists against your own evaluation set in Section 6 — don't ship the defaults untested.


In [13]:
CONF_MIN = 0.6

STOP_KEYWORDS = ["ruko", "ruk jao", "रुको", "रुक जाओ", "stop", "halt"]

def _extract_choice_and_confidence(result: dict):
    """
    Pull the chosen label and its probability out of a Laya `predict` result.

    NOTE: field names below follow the upstream Router.predict shape as
    documented at the time of writing. If your installed `laya` version
    differs, print(result) once and adjust the keys accordingly.
    """
    ans = result["answers"]["command"]
    label = ans["choice"]
    probs = ans.get("probabilities") or ans.get("distribution") or {}
    confidence = probs.get(label, None)
    return label, confidence, probs

def keyword_stop_override(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in STOP_KEYWORDS)

def pipeline(audio_path: str, verbose: bool = False):
    """
    Full audio -> transcript -> command label pipeline.

    Returns a dict with: transcript, label, confidence, raw_result.
    """
    # transcript = transcribe(audio_path)
    transcript="मूव बैग"

    result = router.predict({"utterance": transcript}, COMMAND_QUESTIONS)
    label, confidence, probs = _extract_choice_and_confidence(result)

    if confidence is not None and confidence < CONF_MIN:
        if verbose:
            print(f"Confidence {confidence:.2f} below threshold {CONF_MIN}, forcing 'other'")
        label = "other"

    if keyword_stop_override(transcript) and label != "stop":
        if verbose:
            print("Keyword safety net triggered: overriding to 'stop'")
        label = "stop"

    if verbose:
        print("Transcript :", transcript)
        print("Label      :", label)
        print("Confidence :", confidence)
        print("Full dist. :", probs)

    return {
        "transcript": transcript,
        "label": label,
        "confidence": confidence,
        "raw_result": result,
    }


### 5.1 Try it on a single file

In [14]:
if os.path.exists(TEST_AUDIO_PATH):
    out = pipeline(TEST_AUDIO_PATH, verbose=True)
else:
    print(f"No file found at {TEST_AUDIO_PATH} -- update the path in Section 3.1 and re-run.")


Confidence 0.45 below threshold 0.6, forcing 'other'
Transcript : मूव बैग
Label      : other
Confidence : 0.4543
Full dist. : {'left': 0.0763, 'right': 0.047, 'forward': 0.4543, 'backward': 0.1991, 'stop': 0.0802, 'other': 0.1431}


## 6. Batch inference over a folder of clips

Point this at a directory of audio files (one command per file) to run the pipeline over all of them at once. Useful for a quick spot-check before building a formal evaluation set.


In [ ]:
from pathlib import Path

AUDIO_DIR = Path("path/to/audio_folder")  # <-- change this
AUDIO_EXTS = {".wav", ".flac", ".mp3", ".m4a", ".ogg"}

def run_batch(folder: Path):
    results = []
    if not folder.exists():
        print(f"Folder not found: {folder}")
        return results
    files = sorted(p for p in folder.iterdir() if p.suffix.lower() in AUDIO_EXTS)
    print(f"Found {len(files)} audio files in {folder}")
    for p in files:
        out = pipeline(str(p))
        out["file"] = p.name
        results.append(out)
        print(f"{p.name:40s} -> {out['label']:9s} (conf={out['confidence']})  | '{out['transcript']}'")
    return results

batch_results = run_batch(AUDIO_DIR)


## 7. Evaluation harness

Before trusting this pipeline for anything safety-relevant, measure it against a labeled set. Build a CSV with two columns: `filename` and `true_label` (one of `left`, `right`, `forward`, `backward`, `stop`, `other`), covering both languages, code-mixing, and background noise you'll see in production.

This section computes overall accuracy, a confusion matrix, and per-class precision/recall — and flags `stop` misses specifically, since those are the costliest errors.


In [30]:
import csv

LABELS = ["left", "right", "forward", "backward", "stop", "other"]
LABELED_CSV = "/content/labels.csv"  # <-- change this: columns = filename, true_label
LABELED_AUDIO_DIR = Path("/content/")  # same folder as above, or a separate eval set

def load_labels(csv_path: str):
    rows = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append((row["filename"], row["true_label"].strip().lower()))
    return rows


NameError: name 'Path' is not defined

In [ ]:
def evaluate(csv_path: str, audio_dir: Path):
    if not os.path.exists(csv_path):
        print(f"Labels CSV not found at {csv_path} -- create it first (see markdown above).")
        return None

    rows = load_labels(csv_path)
    y_true, y_pred, records = [], [], []

    for filename, true_label in rows:
        fpath = audio_dir / filename
        if not fpath.exists():
            print(f"Skipping missing file: {fpath}")
            continue
        out = pipeline(str(fpath))
        y_true.append(true_label)
        y_pred.append(out["label"])
        records.append({**out, "file": filename, "true_label": true_label})

    return y_true, y_pred, records

eval_out = evaluate(LABELED_CSV, LABELED_AUDIO_DIR)


### 7.1 Confusion matrix and per-class metrics

In [28]:
import numpy as np
import pandas as pd

def confusion_matrix_df(y_true, y_pred, labels):
    idx = {l: i for i, l in enumerate(labels)}
    mat = np.zeros((len(labels), len(labels)), dtype=int)
    for t, p in zip(y_true, y_pred):
        if t in idx and p in idx:
            mat[idx[t], idx[p]] += 1
    return pd.DataFrame(mat, index=[f"true_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])

def per_class_metrics(y_true, y_pred, labels):
    rows = []
    for l in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == l and p == l)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != l and p == l)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == l and p != l)
        precision = tp / (tp + fp) if (tp + fp) else float("nan")
        recall = tp / (tp + fn) if (tp + fn) else float("nan")
        rows.append({"label": l, "precision": precision, "recall": recall, "support": tp + fn})
    return pd.DataFrame(rows)

if eval_out is not None:
    y_true, y_pred, records = eval_out
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true) if y_true else float("nan")
    print(f"Overall accuracy: {accuracy:.3f} over {len(y_true)} examples\n")

    print("Confusion matrix:")
    display(confusion_matrix_df(y_true, y_pred, LABELS))

    print("\nPer-class precision/recall:")
    display(per_class_metrics(y_true, y_pred, LABELS))

    stop_misses = [r for r in records if r["true_label"] == "stop" and r["label"] != "stop"]
    print(f"\nSTOP commands missed: {len(stop_misses)}")
    for r in stop_misses:
        print(f"  {r['file']}: transcript='{r['transcript']}' predicted='{r['label']}'")


NameError: name 'eval_out' is not defined

### 7.2 Calibration check (optional but recommended)

The Laya documentation is explicit that confidence values need checking, not assuming. This cell buckets predictions by confidence and compares each bucket's average confidence to its observed accuracy — a simple reliability check. If a 0.8-confidence bucket is only 50% correct, your `CONF_MIN` threshold needs to move up.


In [ ]:
def reliability_table(records, n_bins=5):
    scored = [r for r in records if r["confidence"] is not None]
    if not scored:
        print("No confidence values available to calibrate -- check the field-name mapping in Section 5.")
        return None

    bins = np.linspace(0, 1, n_bins + 1)
    rows = []
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        bucket = [r for r in scored if lo <= r["confidence"] < hi or (hi == 1.0 and r["confidence"] == 1.0)]
        if not bucket:
            continue
        avg_conf = np.mean([r["confidence"] for r in bucket])
        acc = np.mean([r["label"] == r["true_label"] for r in bucket])
        rows.append({"bucket": f"[{lo:.1f}, {hi:.1f})", "n": len(bucket), "avg_confidence": avg_conf, "observed_accuracy": acc})
    return pd.DataFrame(rows)

if eval_out is not None:
    _, _, records = eval_out
    table = reliability_table(records)
    if table is not None:
        display(table)


## 8. Next steps

- **Tune `CONF_MIN`** using the reliability table above rather than the placeholder value of 0.6.
- **Expand `STOP_KEYWORDS`** with any additional transliterations or dialect words your users actually say.
- **Watch left/right confusion specifically** — short command words are the most common ASR error class, and it's the costliest one to get wrong if this drives a physical system.
- **Re-run Section 7 whenever you change the prompt, threshold, or either model version** — treat it as a regression test, not a one-time check.
- **Consider shadow-mode deployment**: log predictions alongside the existing control method before letting this pipeline take real actions.
